# 1. Context

This notebook analyzes OCR performance of Tesseract over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

In [2]:
import sys

notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [3]:
from src.viz.helper import display_box_plot

# 2. Extracted Results MetaData

In [4]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese', 'manipuri'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

## 2.1. Language Results Available

In [5]:
results_root = Path("../results/tesseract")

In [6]:
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [7]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Bengali", "Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Devanagari", "Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki", "Devanagari"],
    }

In [8]:
writing_sys_tessract_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_tessract_dict[script].append(language_res)

In [9]:
script_language_result = pd.Series(writing_sys_tessract_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

In [ ]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path

In [11]:
def get_script_results(script: str, script_lang_df: pd.DataFrame):
    """Get results for a given script. output contains results for languages in the script"""

    results_script_path = get_language_results_path(script, script_lang_df)

    results_list = []
    for result_path in results_script_path:
        lang = result_path.parent.name
        df_res = pd.read_csv(result_path)
        df_res['language'] = lang
        results_list.append(df_res)

    results_script = pd.concat(results_list)
    results_script['script'] = script
    return results_script

# 3. Getting All the Results

In [13]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

In [ ]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3))
agg_results.columns = agg_results.columns.str.upper()

In [16]:
agg_results.to_clipboard(index=True) 

In [17]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [18]:
consolidated_df.round(3).to_clipboard(index=False)

# 4. Results Across Various Writing System (samples)

## 4.1. Devanagari

In [19]:
results_devanagari = get_script_results(script='Devanagari', script_lang_df=script_language_result)

### 4.1.1. Box Plot Viz

In [20]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='CER').show()

In [21]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='WER').show()

## 5.1. Bengali

In [22]:
script = 'Bengali'
results_bengali = get_script_results(script='Bengali', script_lang_df=script_language_result)

### 5.1.1. Box Plot Viz

In [23]:
display_box_plot(results_df=results_bengali, script=script, metric_type='CER').show()

# 6. Addendum

## 6.1. Assessing High WER in Hindi

In [24]:
import jiwer

In [29]:
script = 'Devanagari'
results_deva = get_script_results(script=script, script_lang_df=script_language_result)

In [30]:
results_deva_hn = results_deva.loc[results_deva['language'] == 'hindi']

In [37]:
idx = 1
gt = results_deva_hn.loc[idx]['ground_truth']
ocred = results_deva_hn.loc[idx]['ocr_output_L_0']

In [38]:
output_wer = jiwer.process_words(gt, ocred)

In [40]:
print(jiwer.visualize_alignment(output_wer, line_width=10))

=== SENTENCE 1 ===

REF: से
HYP: से
       

REF: बाद
HYP: बाद
        

REF: को
HYP: को
       

REF: साइकिल
HYP: साइकिल
           

REF: को
HYP: को
       

REF: पृथ्वी
HYP:   थ्वी
          S

REF: का
HYP: का
       

REF: चुंबकीय
HYP: चुंबकीय
            

REF: क्षेत्र,
HYP: क्षेत्र,
             

REF: जिसे
HYP: जिसे
         

REF: **
HYP: भू
      I

REF: भू-चुंबकीय
HYP:    चुंबकीय
              S

REF: क्षेत्र
HYP: क्षेत्र
            

REF: के
HYP: के
       

REF: रूप
HYP: रूप
        

REF: में
HYP: में
        

REF: भी
HYP: भी
       

REF: जाना
HYP: जाना
         

REF: जाता
HYP: जाता
         

REF: है।
HYP: है।
        

REF: वही
HYP: वही
        

REF: चुंबकीय
HYP: चुंबकीय
            

REF: क्षेत्र
HYP: क्षेत्र
            

REF: है जो
HYP: है जो
          

REF: पृथ्वी
HYP: पृथ्वी
           

REF: के
HYP: के
       

REF: आंतरिक
HYP: आंतरिक
           

REF: भाग
HYP: भाग
        

REF: में
HYP: में
        

REF: से
HYP: सै
      S

REF: अंतरिक्ष
HYP: अंतरिक्ष
    

In [42]:
print(jiwer.visualize_error_counts(output_wer))

=== SUBSTITUTIONS ===
पृथ्वी            --> थ्वी              = 1x
भू-चुंबकीय        --> चुंबकीय           = 1x
से                --> सै                = 1x
ये                --> रे                = 1x
पवन(solar wind)   --> पवन(50]97' ४एं70) = 1x
पृथ्वी            --> जवां              = 1x
"इन               --> कर                = 1x
गति"              --> गति.              = 1x
हैं।"             --> "                 = 1x
धुरी              --> धर                = 1x
धाराओं            --> धारा:             = 1x
होता              --> उनपर              = 1x
करती              --> करंती             = 1x
परिचय.            --> परिचय,            = 1x
अर्थात            --> मथति              = 1x
चुंबकीय           --> [बकीय             = 1x
मापन              --> मापन्‌            = 1x
(D), नति (I),     --> 0 न ([),          = 1x
पार्थिव           --> पार्थवि           = 1x
(F),              --> 7),               = 1x
(H), (X), (Y)     --> ( (जु), (४)       = 1x
(Z)               --> (7)        

In [43]:
print(gt)

से बाद को साइकिल को पृथ्वी का चुंबकीय क्षेत्र, जिसे भू-चुंबकीय क्षेत्र के रूप में भी जाना जाता है। वही चुंबकीय क्षेत्र है जो पृथ्वी के आंतरिक भाग में से अंतरिक्ष में फैलता है। जहां ये सूर्य से आ रही सौर पवन(solar wind) और चार्ज कणों से पृथ्वी को बचाता है। "इन कणों  की गति" पृथ्वी के "चुंबकीय क्षेत्र से प्रभावित होती है। और यह स्वयं भी पृथ्वी के चुंबकीय क्षेत्र व्यवस्था मैं परिवर्तन ला सकते हैं।" पृथ्वी के बाहरी कोर में पिघले हुए लोहे और निकल के मिश्रण की संवहन धाराओं की गति के कारण विद्युत धाराओं उत्पन्न होती है, और पृथ्वी के अपनी धुरी पर घूमने के कारण वो विद्युत धाराओं में से चुंबकीय क्षेत्र उत्पन्न होता है। यह प्रक्रिया कुछ वैसे ही काम करती है जैसे  साइकिल पर डायनेमो लाइट। जब साइकिल को चलाया जाता है, तब डायनेमो में मौजूद चुम्बक घूमने लगते हैं। जिसकी वजह है विद्युत प्रवाह उत्पन होता है। जिस से बल्ब जलाया जाता है। अब अगर बल्ब की जगह विद्युत प्रवाह को एक जगह पर रोटेट किया जाय तो वो एक विद्युत चुंबक बनाए जाएगा। ठीक वैसे ही पृथ्वी का चुंबकीय क्षेत्र उत्पन्न होता है। अब ऊपर हमने जिस संवहन 

In [44]:
print(ocred)

से बाद को साइकिल को  थ्वी का चुंबकीय क्षेत्र, जिसे भू चुंबकीय क्षेत्र के रूप में भी जाना जाता है। वही चुंबकीय क्षेत्र है जो पृथ्वी के आंतरिक भाग में सै अंतरिक्ष में फैलता है। जहां रे सूर्य से आ रही सौर पवन(50]97' ४एं70) और चार्ज कणों से जवां को बचाता है।  गन कर कणों की गति. पृथ्वी के "चुंबकीय क्षेत्र से प्रभावित होती है। और यह स्वयं भी पृथ्वी के चुंबकीय क्षेत्र व्यवस्था मैं ला सकते हैं। "  पृथ्वी के बाहरी कोर में पिघले हुए लोहे और निकल के मिश्रण की संवहन धाराओं की गति के कारण विद्युत धाराओं उत्पन्न होती है, और पृथ्वी के अपनी धर पर घूमने के कारण वो विद्युत धारा: चुंबकीय क्षेत्र उत्पन्न उनपर प्रक्रिया कुछ वैसे ह ही काम करंती है जैसे पर डायनेमो लाइट। जब साइकिल को चलाया जाता है, तब चुम्बक हैं। जिसकी वजह है विद्युत प्रवाह उत्पन होता है। जिस से बल्ब जलाया जाता है। अब अगर बल्ब की जगह प्रवाह को एक जगह पर रोटेट किया जाय तो वो एक विद्युत त चुंबक बनाए जाएगा। ठीक वैसे ही पृथ्वी का चुंबकीय क्षेत्र उत्पन्न है। अब ऊपर हमने जिस संवहन धाराओं कि बात की वो संवहन धाराएँ कोर से निकलने वाली गर्मी से उत्पन्न